# Access Sentinel-2 imagery via the STAC api

This notebook provides a step-by-step guide on how to use the Copernicus Data Space Ecosystem STAC (SpatioTemporal Asset Catalog) API to search for Sentinel-2 Level 2A data within a specified time range and geographic area. It demonstrates how to make use of the STAC extensions for filtering, selecting, and sorting data based on specific parameters.


## Libraries

First, ensure all required packages are installed and imported.

In [0]:
%pip install --upgrade stackstac && pip install folium pystac_client geogif s3fs stacmap shapely rasterio


In [0]:
dbutils.library.restartPython()


In [0]:
import os
from pathlib import Path
import shutil
import urllib.parse
from urllib.parse import urlparse
from collections.abc import Mapping
import subprocess
import pandas as pd
from shapely.geometry import shape
import folium
from folium.plugins import Draw
import xarray as xr
import fsspec
import geogif
import json
import pystac_client
import requests
import s3fs
import stackstac
#import stacmap
import rasterio

* `pystac_client`, `stackstack`: A Python library for working with STAC catalogs. It enables interaction with STAC services, reading catalog information, and searching for products based on specified parameters.
* `geogif`: A Python library for creating animated GIFs from data cubes, enabling the visualization of time series data in a dynamic and engaging format.
* `os`: A standard Python package that provides functionality for interacting with the operating system, including accessing environment variables and managing file paths. It is often used for authentication and setting up access credentials for various APIs and services, such as STAC.

## S3 Credentials

In order to access the data from CloudFerro's catalog, authentication is required. To facilitate this, you will need to create an account in the Copernicus Data Space Ecosystem (CDSE). The registration process is quick and can be completed in under 60 seconds. You can find the registration link here: https://documentation.dataspace.copernicus.eu/APIs/S3.html.

Once you have created an account, the same URL will redirect you to the section where you can generate your `S3 Credentials`. To do this, simply click the `Add Credential` button. This will allow you to define an expiration date for your credentials.

After selecting the expiration date, you will have successfully created your own S3 credentials. Please make sure to save the secret key at this point, as it will not be displayed again.

Now that you have your S3 credentials, you can use them to connect to the STAC catalog. By utilizing the `os` package, you will set environment variables to authenticate yourself as a valid user for the S3 protocol.

The only variables you need to configure are:
* `AWS_ACCESS_KEY_ID`
* `AWS_SECRET_ACCESS_KEY`.

For this we will use [Databricks Secrets](https://learn.microsoft.com/en-us/azure/databricks/security/secrets/)

In [0]:
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = dbutils.secrets.get(scope = "tomkdefra_cdse", key = "AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope = "tomkdefra_cdse", key = "AWS_SECRET_ACCESS_KEY")
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

## Connecting to the STAC Catalog

We connect to the Copernicus STAC catalog, available at the endpoint: https://stac.dataspace.copernicus.eu/v1.

In [0]:
URL = "https://stac.dataspace.copernicus.eu/v1"
cat = pystac_client.Client.open(URL)
cat.add_conforms_to("ITEM_SEARCH")

## Defining the Area of Interest (AOI)

We define the area as a polygon based on geographical coordinates in GeoJSON format. We are going to get these interactively using a map widget in the notebook. Running the next cell will load a map widget, with a drag to draw polygon feature enabled. Once you have drawn a polygon, left-click the 'Export' button to activate the GeoJSON download, and then right-click the same button and chose 'copy link'.

* `type`: Geometry type, in this case, a polygon.
* `coordinates`: A list of coordinates defining the area.

In [0]:
center = [52.561928, -1.464854]
zoom = 7
m = folium.Map(location=center, zoom_start=zoom)
draw = Draw(
    export=True,
    draw_options={
        'polyline': False,
        'circlemarker': False,
        'polygon': False,
        'circle': False,
        'rectangle': True,
        'marker': True,
        'edit': True
    },
)

draw.add_to(m)
display(m)

def extract_polygon_coords(polygon_export_link):
    try:
        json_string = polygon_export_link.split(',', 1)[1]
        decoded_json = urllib.parse.unquote(json_string)
        geojson = json.loads(decoded_json)

        return geojson['features'][0]['geometry']['coordinates']
    except (IndexError, KeyError, json.JSONDecodeError) as e:
        raise ValueError("Invalid link. Please activate the link first by left-clicking the 'Export' button, before copying the link.") from e


### Extracting the coordinates

Paste the link you just copied in place of the one which begins with "data:text/json" in the next cell, then run the cell. The extracted coordinates will be passed to the rest of the notebook. 

In [0]:
polygon_export_link = "data:text/json;charset=utf-8,%7B%22type%22%3A%22FeatureCollection%22%2C%22features%22%3A%5B%7B%22type%22%3A%22Feature%22%2C%22properties%22%3A%7B%7D%2C%22geometry%22%3A%7B%22type%22%3A%22Polygon%22%2C%22coordinates%22%3A%5B%5B%5B-5.146426%2C51.568534%5D%2C%5B-5.146426%2C51.706183%5D%2C%5B-4.62466%2C51.706183%5D%2C%5B-4.62466%2C51.568534%5D%2C%5B-5.146426%2C51.568534%5D%5D%5D%7D%7D%5D%7D"

# Decode the URL string
encoded_json = polygon_export_link.split(',', 1)[1]
decoded_json_str = urllib.parse.unquote(encoded_json)

# Parse the GeoJSON FeatureCollection
feature_collection = json.loads(decoded_json_str)
aoi_feature = feature_collection['features'][0]
aoi_geometry = aoi_feature['geometry']

# Create a Shapely object and extract the WGS84 bounding box (min_lon, min_lat, max_lon, max_lat)
aoi_polygon = shape(aoi_geometry)
aoi_bounds_latlon = aoi_polygon.bounds
print(f"BBOX for stackstac: {aoi_bounds_latlon}")

# Prepare the geometry for the STAC search
geom_for_search = {
    "intersects": aoi_geometry
}

## Defining Search Parameters

In this section, we define and combine parameters into a single dictionary for passing to the search function:
* `max_items`: The maximum number of results to retrieve.
* `collections`: The STAC collection (e.g., Sentinel-2 Level 2A data).
* `datetime`: Time range in YYYY-MM-DD/YYYY-MM-DD format.
* `intersects`: The previously defined area of interest.
* `query`: Typically a key-value pair where the key represents a specific property (e.g., eo:cloud_cover), and the value is the condition or comparison being applied.
* `sortby`: Sorts results (e.g., by the eo:cloud_cover property in ascending order).
* `fields`: Specifies which fields to exclude (e.g., geometry).

In [0]:
# List all available collections (catalogs) in the STAC API
collections = cat.get_collections()
collections_data = [{"id": c.id, "title": c.title, "description": c.description} for c in collections]

display(collections_data)

## Searching the Catalog

We query the catalog and convert the results into a list of dictionaries.

In [0]:
DATE_RANGE = "2025-05-01/2025-10-30"

search_params = {
    "max_items": 100,
    "collections": "sentinel-2-l2a",  # Paste the collection id from the table above
    "datetime": DATE_RANGE,
    **geom_for_search,
    "query": {
        "eo:cloud_cover": {"lte": 10},
    },
}
items = list(cat.search(**search_params).items())
print(f"Found {len(items)} items matching your search parameters.")

## Now we can use StacMap to view some of our results

In [0]:
stacmap.explore(
    items, 
    prop="eo:cloud_cover",
    thumbnails=True
)


### Display the Asset Keys for First Item
This cell iterates over the first item in the `items` list and prints the asset keys. You can use these keys to download or access different items in the catalogue. Taking the red band 04 as an example, the STAC api provides access to three items for that band: the native 10m resolution data, as well as resampled data at 20m and 60m resolutions.



In [0]:
item = next(iter(items), None)

if item is not None:
    assets = getattr(item, "assets", {})
    keys = list(assets.keys()) if isinstance(assets, Mapping) else []
    print(f"Item {item.id} asset keys: {keys}")


## Create a datacube of item assets and HREF attributes


In [0]:
selected_assets = ["B04_10m", "B08_10m"]

In [0]:
cube = stackstac.stack(
    items, 
    assets=selected_assets,     
    bounds_latlon=aoi_bounds_latlon,
    epsg=27700,  # British National Grid
    resolution=10,  # Native 10m resolution of the red and NIR bands of Sentinel-2
    chunksize=4096
)

print(f"Data Cube successfully created in EPSG:27700 (meters) with shape: {cube.shape}")
display(cube)

In [0]:
# Take a band asset from the first STAC item in the list for testing.
href = items[0].assets["B04_10m"].href
print(href)
